## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# 100 binomial draws, five times: sample against theory

**Random Number Generation** · a worked example from the StatNex teaching guide

**The story.** A simulation starts with random numbers. This run draws them from a binomial distribution: each number counts the successes in 10 tries, where each try succeeds half the time. It makes 100 numbers, five times over, with a fixed seed. It uses no column of the table.

**The question.** How close do the mean and the variance of simulated data come to their theoretical values?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `12_utility/matrix_inverse.csv` (5 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Measure,Meetings per week,Emails per day,Deals closed,Revenue ($000),Customer rating
Meetings per week,1.0,0.3586,0.7429,0.7164,0.1357
Emails per day,0.3586,1.0,0.1832,0.1392,-0.2765
Deals closed,0.7429,0.1832,1.0,0.9462,0.149
Revenue ($000),0.7164,0.1392,0.9462,1.0,0.1354
Customer rating,0.1357,-0.2765,0.149,0.1354,1.0
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## Settings

In [ ]:
# count numbers in each of `columns` columns. The generator is numpy's default (PCG64),
# the same one the app uses, so with the same seed you get the app's numbers exactly
# (numpy may change a distribution's algorithm between releases, so that holds for
# the numpy version the app used). Without a seed (None) every run gives new numbers.
count = 100
columns = 5
seed = 20261027
rng = np.random.default_rng(seed)

## Draws

In [ ]:
# Binomial: successes in a trials with success probability b. Mean ab, variance ab(1 - b).
a, b = float(10), float(0.5)
draws = rng.binomial(int(a), b, (count, columns)).astype(float)
mean, variance = a * b, a * b * (1 - b)
labels = [str(i + 1) for i in range(columns)]
theory = pd.Series({"mean": mean, "variance": variance})

## Theoretical mean and variance

In [ ]:
print(theory)

## Sample statistics of each column

In [ ]:
# The variance here is the unbiased one, dividing by n - 1 (ddof=1), as the app and the
# covariance matrix below do. Each column is one sample, so its statistics scatter
# around the theoretical values: about 95% of the column means should land within
# 2 * sqrt(variance / count) of the theoretical mean. Further out, suspect the
# settings, not the generator.
sample_statistics = pd.DataFrame([draws.mean(axis=0), draws.var(axis=0, ddof=1)],
                                 index=["mean", "unbiased variance"], columns=labels)
print()
print(sample_statistics)

## The numbers

In [ ]:
table = pd.DataFrame(draws, index=pd.RangeIndex(1, count + 1, name="No."), columns=labels)
table.to_csv("random_numbers.csv")
print()
print(table)

## What the result says

Each number is a draw from a binomial distribution: the count of successes in 10 tries, when each try succeeds with probability 0.5. The run makes 100 such numbers in each of 5 columns. In theory the mean is 5 and the variance, a measure of spread, is 2.5.

No column hits these values exactly. The five column means run from 4.84 to 5.33, and the unbiased variances, the sample estimates of the variance, run from 2.40 to 2.85.

These exact numbers come from the seed, the starting value of the random number generator. Another seed gives other numbers, but the lesson stays: simulated data hit their targets only on average. With 100 draws, a gap of a few tenths from the theoretical mean is normal, so do not read meaning into it.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.